In [1]:
!pip install medmnist


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.5 MB/s eta 0:00:00


In [2]:
from torch.utils.data import DataLoader

from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import medmnist
import pandas as pd
from medmnist import INFO, Evaluator
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import torch

In [3]:
data_flag = 'chestmnist'
# data_flag = 'breastmnist'
download = True


info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])
# preprocessing
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),  # 🔥 1 → 3 channels
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])
# load the data

corpus = DataClass(split='test', transform=data_transform, download=True)

X = []
Y = []

for img, label in corpus:
    X.append(img.numpy())
    Y.append(label)

X= np.array(X)
Y = np.array(Y)

100%|██████████| 82.8M/82.8M [00:05<00:00, 16.1MB/s]


In [4]:
print(len(DataClass(split='train', transform=data_transform, download=True)))

78468


In [5]:
print(len(corpus))

22433


In [6]:
loader = DataLoader(
    corpus,
    batch_size=32,  #bigger migth overflow dino
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# RADDINO 

In [7]:


#processor = AutoImageProcessor.from_pretrained("microsoft/rad-dino")
model = AutoModel.from_pretrained("microsoft/rad-dino")



config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [9]:
features_corset = []
labels_corset = []

with torch.no_grad():
    for imgs, lbls in loader:
        #print("o")
       # inputs = processor(images=imgs, return_tensors="pt")
        #inputs = {k: v.to(device) for k, v in inputs.items()}

       # outputs = model(**inputs) #instead of usingproce cause you already did el procesing with transofrm
        outputs = model(pixel_values=imgs.to(device))
        features = outputs.last_hidden_state.mean(dim=1) 
        features_corset.append(features)
        labels_corset.append(lbls)
        if len(features_corset)%50==0:
            print(len(features_corset)/len(loader))
        

0.07122507122507123
0.14245014245014245
0.21367521367521367
0.2849002849002849
0.3561253561253561
0.42735042735042733
0.4985754985754986
0.5698005698005698
0.6410256410256411
0.7122507122507122
0.7834757834757835
0.8547008547008547
0.9259259259259259
0.9971509971509972


In [10]:
print(len(features_corset),len(labels_corset))

702 702


In [11]:
import numpy as np
import random
from sklearn.neighbors import NearestNeighbors

class GreedyCoreset:
    def __init__(self, corpus, labels=None):
        self.corpus = np.array(corpus)
        self.labels = labels
        self.n = len(corpus)
        
        # you use nn later to get the 100 nearest neighbors for each dp
        self.nn = NearestNeighbors(n_neighbors=min(100, self.n), metric='euclidean')
        self.nn.fit(self.corpus)
    
    def select(self, K=50, sample_size=100, per_label=False): #what you cocall
        if per_label and self.labels is not None:
            return self._select_per_label(K, sample_size)
        else:
            return self._select(K, sample_size)
    
    def _select(self, K, sample_size):  #the core corset selection algo 
        C = []
        
        # array to keep for each datapoint in corpus the min distance we have from a point to it
        #initialse with sth huge
        min_dists = np.full(self.n, np.inf)
        """
        min_dist is storing for each point the closest representative to it from points from C
        """
        
        # First point: pick farthest from center or random
        if K > 0:
            # Pick point farthest from mean (diverse start)
            mean = np.mean(self.corpus, axis=0)
            first_idx = np.argmax([np.linalg.norm(x - mean) for x in self.corpus])
            C.append(first_idx)
            
            # Update min distances
            for i in range(self.n):
                min_dists[i] = np.linalg.norm(self.corpus[i] - self.corpus[first_idx])
        
        while len(C) < K:
            # Sample candidates (not in C)
            """ candidates = list(set(range(self.n)) - set(C))
            if len(candidates) > sample_size:
                candidates = random.sample(candidates, sample_size)"""
            candidates=get_samples(self.corpus, C, size_samples=100)
            best_t = None
            best_utility = -np.inf
            
            for t in candidates:
                # Fast utility computation using nearest neighbors
                # Get distances from t to all points (approximate)
                dist_to_t, indexes = self.nn.kneighbors(
                    self.corpus[t].reshape(1, -1), 
                  #  n_neighbors=self.n,  #why specify n == everyone 
                    return_distance=True
                )
                dist_to_t = dist_to_t[0]
                indexes = indexes[0]

                # Compute reduction
                reduction = 0
                l_neighbors=len(dist_to_t)
                """
                amoung the k closest neighbors we picked we are gonna see 
                which of them is the cloest to the rest of datapoitns
                from the main corpus 
                """
                for j, i in enumerate(indexes): # i is dataset index  and j positiion in neigh list 
                    if i in C:
                        continue
                    if dist_to_t[j] < min_dists[i]:
                        #this bewlo is teh utility
                        # E(c) -E[c+t]
                        reduction += min_dists[i] - dist_to_t[j]
                #reduction is the "imporvment " it will bring to the corset
                #
                if reduction > best_utility:
                    best_utility = reduction
                    best_t = t
            
            if best_t is not None:
                C.append(best_t)
                
                # Update min distances
                for i in range(self.n):
                    if i in C:
                        continue
                    """
                    here you added new point t to the corset , and remember that mindist 
                    is computing dis based on points in the corset 
                    since you added new point to corset recompute to see 
                    """
                    #PP
                    dist = np.linalg.norm(self.corpus[i] - self.corpus[best_t])
                    if dist < min_dists[i]:
                        min_dists[i] = dist
            
            print(f"Selected {len(C)}/{K} tuples")
        
        return C
    
    def _select_per_label(self, K, sample_size):
        unique_labels = np.unique(self.labels)  #get all of out labels here they are 
        #update per the frequency of the class
        
        C = []
        
        for label in unique_labels:
            label_indices = np.where(self.labels == label)[0]
            label_ratio = len(label_indices) / self.n
            label_K = max(1, int(K * label_ratio))
            
            if label_K > 0 and len(label_indices) > 0:
                # Create sub-selector for this label
                label_selector = GreedyCoreset(
                    self.corpus[label_indices], 
                    labels=None
                )
                label_C = label_selector._select(label_K, sample_size)
                
                # Map back to original indices
                C.extend(label_indices[label_C].tolist())
        
        return C
    
    def compute_weights(self, C):
        """Compute weights for coreset points"""
        weights = np.zeros(len(C))
        
        for i in range(self.n):
            # Find closest coreset point
            min_dist = np.inf
            closest_idx = -1
            for j, c in enumerate(C):
                dist = np.linalg.norm(self.corpus[i] - self.corpus[c])
                if dist < min_dist:
                    min_dist = dist
                    closest_idx = j
            
            weights[closest_idx] += 1
        
        return weights



In [17]:
saved_samples=[]
import random

def get_samples(corpus, C, size_samples=100):
    candidates = list(set(range(len(corpus))) - set(C))
    if len(candidates) <= size_samples:
        return candidates
    return random.sample(candidates, size_samples)

In [13]:
np.unique([0,1,0],[1,0,0])

(array([0, 1]), array([0, 1]))

In [14]:
features_corset = torch.cat(features_corset, dim=0)
labels_corset = torch.cat(labels_corset, dim=0)

In [23]:
print(len(features_corset),len(labels_corset))

22433 22433


In [19]:
features_corset.cpu()
labels_corset.cpu()

tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 1,  ..., 0, 0, 0],
        [0, 0, 1,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]])

In [21]:
features_corset = features_corset.cpu().numpy()
labels_corset = labels_corset.cpu().numpy()

selector = GreedyCoreset(features_corset, labels_corset)

In [25]:
coreset_indices = selector.select(K=1120, sample_size=100)


Selected 2/1120 tuples
Selected 3/1120 tuples
Selected 4/1120 tuples
Selected 5/1120 tuples
Selected 6/1120 tuples
Selected 7/1120 tuples
Selected 8/1120 tuples
Selected 9/1120 tuples
Selected 10/1120 tuples
Selected 11/1120 tuples
Selected 12/1120 tuples
Selected 13/1120 tuples
Selected 14/1120 tuples
Selected 15/1120 tuples
Selected 16/1120 tuples
Selected 17/1120 tuples
Selected 18/1120 tuples
Selected 19/1120 tuples
Selected 20/1120 tuples
Selected 21/1120 tuples
Selected 22/1120 tuples
Selected 23/1120 tuples
Selected 24/1120 tuples
Selected 25/1120 tuples
Selected 26/1120 tuples
Selected 27/1120 tuples
Selected 28/1120 tuples
Selected 29/1120 tuples
Selected 30/1120 tuples
Selected 31/1120 tuples
Selected 32/1120 tuples
Selected 33/1120 tuples
Selected 34/1120 tuples
Selected 35/1120 tuples
Selected 36/1120 tuples
Selected 37/1120 tuples
Selected 38/1120 tuples
Selected 39/1120 tuples
Selected 40/1120 tuples
Selected 41/1120 tuples
Selected 42/1120 tuples
Selected 43/1120 tuples


In [26]:
import numpy as np

np.save("coreset_indices_test.npy", coreset_indices)

In [27]:
print(coreset_indices)

[np.int64(12988), 16951, 19695, 13367, 10859, 20052, 8719, 4536, 12305, 20320, 4765, 7752, 2125, 15387, 9871, 6835, 15608, 21941, 18758, 14387, 4233, 9013, 11252, 1677, 741, 22102, 14209, 9250, 19840, 13838, 9408, 21559, 18732, 10683, 5170, 5200, 21449, 5671, 12109, 12766, 21591, 11301, 21014, 11974, 11623, 15967, 15738, 11138, 4131, 14389, 8678, 9597, 8037, 14286, 13866, 8072, 21866, 186, 11563, 18330, 9626, 5738, 8321, 16557, 21045, 12612, 9499, 6077, 12827, 13326, 10339, 11236, 21428, 19419, 20742, 9121, 11860, 3107, 15912, 7915, 4972, 6195, 2828, 4515, 760, 11890, 13653, 5222, 7305, 1114, 22035, 13765, 19526, 21319, 19553, 12460, 11685, 1687, 21600, 4085, 13835, 13537, 12815, 16487, 20103, 11568, 7219, 17622, 6383, 17424, 7438, 17083, 16467, 1919, 4190, 21436, 18637, 7855, 11402, 6730, 14753, 129, 17072, 455, 15183, 2138, 16505, 20756, 6672, 1698, 652, 1626, 4188, 18963, 20931, 749, 12505, 2824, 13151, 11918, 17993, 10059, 14157, 21440, 18895, 19918, 14598, 9523, 6138, 7083, 14810,

In [30]:
print(max(coreset_indices),min(coreset_indices),len(coreset_indices))

22421 0 1120
